# CATLA — Phase 6 LoRA fine-tuning (Google Colab, free T4)

Trains one ensemble engine at a time (IndicTrans2 / NLLB-200-distilled-600M / BanglaT5), both directions (bn2en, en2bn), via `pipeline/train.py`. This notebook cannot run unattended for hours — free Colab sessions require an active browser tab and disconnect after inactivity/~12h — so it checkpoints every `--save_steps` and auto-resumes from the latest checkpoint on re-run.

**Prerequisite**: Phase 5's `pipeline/push_to_hf.py` must have already been run (by you, on your own machine, after `huggingface-cli login`) so `train.jsonl`/`val.jsonl` are available from the Hub — this notebook does not redo Phases 1-5, it only fetches their output.

Set `HF_DATASET_REPO` and `HF_MODEL_REPO_PREFIX` below before running.

In [ ]:
!nvidia-smi

In [ ]:
HF_DATASET_REPO = "CHANGE_ME/catla-bn-en-tweets"  # from Phase 5's push_to_hf.py
HF_MODEL_REPO_PREFIX = "CHANGE_ME/catla"  # LoRA adapters pushed as <prefix>-<engine>-<direction>
GITHUB_REPO = "https://github.com/Shanjiv931/nlp_caslf.git"

ENGINES = {
    "indictrans2": "ai4bharat/indictrans2-en-indic-1B",  # swap for the -200M distilled variant if this OOMs on T4
    "nllb": "facebook/nllb-200-distilled-600M",
    "banglat5": "csebuetnlp/banglat5",
}
DIRECTIONS = ["bn2en", "en2bn"]

In [ ]:
!git clone $GITHUB_REPO caslf
%cd caslf
!pip install -q -r requirements.txt
!pip install -q IndicTransToolkit  # AI4Bharat's recommended IndicTrans2 preprocessing; train.py auto-detects it
!pip uninstall -y torchao -q  # Kaggle/Colab base images can ship a torchao version too old for peft's
# LoRA dispatch check, which raises ImportError instead of skipping gracefully. We don't use
# torchao at all for plain LoRA, so removing it entirely is the safe fix.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # paste your own free HF token when prompted — never hard-code it in this notebook

In [ ]:
from datasets import load_dataset
import os, json

ds = load_dataset(HF_DATASET_REPO)
os.makedirs("data/processed", exist_ok=True)
for split, fname in [("train", "train.jsonl"), ("val", "val.jsonl"), ("test", "test.jsonl")]:
    ds[split].to_json(f"data/processed/{fname}", force_ascii=False)
print("data ready:", {k: len(v) for k, v in ds.items()})

## Train one engine + direction at a time
Re-run this cell (changing `engine_key`/`direction`) for each of the 6 combinations. `--resume` is safe to pass even on a first run — it's a no-op if no checkpoint exists yet.

In [ ]:
engine_key = "nllb"  # one of: indictrans2, nllb, banglat5
direction = "bn2en"  # or en2bn

model_name = ENGINES[engine_key]
output_dir = f"/content/drive/MyDrive/catla_checkpoints/{engine_key}_{direction}" if os.path.exists("/content/drive") else f"./checkpoints/{engine_key}_{direction}"
hub_repo = f"{HF_MODEL_REPO_PREFIX}-{engine_key}-{direction}"

# Restrict to a single GPU. On a T4 x2 session, transformers' Trainer
# auto-wraps the model in naive DataParallel across both GPUs, but
# DataParallel funnels loss computation + gradient-gathering through GPU 0
# specifically, which can OOM that one GPU even with plenty of *combined*
# memory across both (hit this for real on 2026-08-05, mid-training).
# Single-GPU sidesteps the imbalance entirely rather than tuning around it.
!CUDA_VISIBLE_DEVICES=0 python pipeline/train.py \
  --model_name "$model_name" \
  --direction "$direction" \
  --output_dir "$output_dir" \
  --epochs 1 \
  --max_train_rows 150000 \
  --batch_size 8 \
  --grad_accum 4 \
  --save_steps 500 \
  --resume \
  --push_to_hub "$hub_repo"

### Recommended: mount Google Drive first so checkpoints survive a session disconnect
```python
from google.colab import drive
drive.mount('/content/drive')
```
Run this before the training cell above — `output_dir` auto-detects `/content/drive` and checkpoints there instead of the ephemeral local disk.